# Demo: Building a RAG-powered FAQ Agent with Custom Knowledge

# Step 1: Install required packages

In [1]:
!pip install -U langchain langchain-openai langchain-community langchain-classic faiss-cpu tiktoken

# Step 2: Import dependencies

In [2]:
import os
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA
#update

/tmp/ipykernel_14902/662839124.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


# Step 3: Set Azure OpenAI credentials



In [3]:
os.environ["AZURE_OPENAI_API_KEY"] = "2ABecnfxzhRg4M5D6pBKiqxXVhmGB2WvQ0aYKkbTCPsj0JLKsZPfJQQJ99BDAC77bzfXJ3w3AAABACOGi3sC"

# Step 4: Load and chunk your custom FAQ document


In [4]:
# Load the FAQ document and split it into chunks for embedding

loader = TextLoader("faq.txt")  # Ensure this file exists
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

# Step 5: Create vectorstore using Azure embeddings


In [5]:
# Embed document chunks and store them in a FAISS vector index

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint="https://openai-api-management-gw.azure-api.net",
    api_version="2023-05-15",
    deployment="text-embedding-ada-002",
    api_key=os.environ["AZURE_OPENAI_API_KEY"]
)

vectorstore = FAISS.from_documents(docs, embeddings)

# Step 6: Initialize the Azure OpenAI LLM

In [6]:
# Initialize the GPT-4o model from Azure with temperature 0 for deterministic output

llm = AzureChatOpenAI(
    azure_endpoint="https://openai-api-management-gw.azure-api.net",
    api_version="2025-01-01-preview",
    deployment_name="gpt-5-mini"
)
#update

# Step 7: Create the RAG chain

In [7]:
# Create a RetrievalQA chain that uses the retriever and LLM to answer queries with source context

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    return_source_documents=True
)

# Step 8: Ask a question

In [14]:
# Send a question to the RAG chain and store the result

query = "After Return of defective goods, how will i receive the refund and when i will receive the refund"
result = qa_chain.invoke({"query": query})

# Step 9: Print results

In [15]:
# Display the final answer and the source chunks used

print("Answer:", result["result"])
print("\n--- Sources ---")
for i, doc in enumerate(result["source_documents"], 1):
    print(f"\nSource {i}:")
    print(doc.page_content)

Answer: For defective items the policy in our notes is: contact support within 48 hours of delivery to request a replacement or refund. Please have your order number, photos of the defect, and proof of delivery ready when you contact them.

I don’t have the exact details here about the refund method or the exact timeframe for processing refunds. Please contact customer support and they will confirm how and when the refund will be issued:

- Email: support@example.com  
- Phone: +1-800-123-4567 (9AM–6PM, Mon–Fri)

If you want, tell me your order number and a brief description of the defect and I can suggest what to include in your message to support.

--- Sources ---

Source 1:
Q: Can I track my order?
A: Yes, once your order is shipped, you will receive a tracking link via email and SMS.

Q: What should I do if I receive a damaged product?
A: If the item is damaged or defective, please contact our support team within 48 hours of delivery for a replacement or refund.

Q: Are there any d